# MLflow Eval 📊

We would like to be able to evaluate our models and applications before they go into production.  
To do that we can use MLflow's evaluation framework 🙌  
It allows us to run prompts and expected answers through different scorers to make sure that the model answers as we expect.  
The prompts and expected answers are stored in an MLflow-managed dataset, which lets us version and track our evaluation data alongside our experiments over time.

We will be testing two types of evaluations here:
- A **simple/functional scorer** (`is_concise`) which tests if the LLM output stays within a character limit. No extra LLM calls required, fast and cheap to run
- An **LLM as judge** scorer (`summary_quality`) which lets an LLM evaluate how well the generated answer addresses the conversation

We will evaluate the backend endpoint so that we also test how effective our system prompts are in practice 😉

In [ ]:
!pip install mlflow

## Set-up
Let's start by setting up our clients. We use an OpenAI-compatible client pointing to our vLLM-hosted Llama 3.2 model, and configure MLflow to track evaluation results against our tracking server.

In [ ]:
import mlflow
from mlflow.entities import AssessmentSource, AssessmentSourceType
from mlflow.genai.scorers import scorer
from mlflow.genai.judges import make_judge
from openai import OpenAI
from typing import Literal
import os, re, json

In [ ]:
LLM_ENDPOINT = "http://llama-32-predictor.ai501.svc.cluster.local:80"
MODEL_NAME = "llama32"

client = OpenAI(
    base_url=LLM_ENDPOINT + "/v1",
    api_key="no-key-required", #for now
)

Now let's point MLflow to our tracking server. We also set the experiment name so all our evaluation runs are grouped together.

In [ ]:
MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

NAMESPACE_PATH = "/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(NAMESPACE_PATH):
    with open(NAMESPACE_PATH) as f:
        os.environ["MLFLOW_WORKSPACE"] = f.read().strip()

SA_TOKEN_PATH = "/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("summarization")

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Workspace: {os.environ.get('MLFLOW_WORKSPACE', 'not set')}")

# Datasets 📖

We use an MLflow-managed dataset to store our evaluation data. This lets us version it, reuse it across runs, and track exactly which data each evaluation was run against.  
We fetch the existing dataset and merge in new samples. Each sample has `inputs` (the conversation messages) and `expectations` (the criteria we'll score against, such as a maximum character length).

## Add samples to the dataset

In [ ]:
eval_dataset = mlflow.genai.get_dataset(
    name="eval_dataset",
)

In [ ]:
new_samples = [                                                                                                                                                                                                                        
    {                                                                                                                                                                                                                                  
        "inputs": {                                                                                                                                                                                                                  
            "messages": [
                {"role": "user", "content": "Artificial intelligence is transforming healthcare by enabling faster diagnostics and personalized treatment plans."},
            ],
            "session_id": "session-001",
        },
        "expectations": {"length": 150},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "The Eiffel Tower was built in 1889 and stands 330 metres tall in Paris, France."},
                {"role": "assistant", "content": "The Eiffel Tower, built in 1889, is a 330-metre landmark in Paris. 🗼"},
                {"role": "user", "content": "What year was it built again?"},
            ],
            "session_id": "session-002",
        },
        "expectations": {"length": 100},
    },
]

eval_dataset.merge_records(new_samples)
print(f"Dataset now has records: {eval_dataset.to_dict()['profile']}")

The dataset now contains our evaluation samples, each pairing an input with an expected answer. MLflow handles storing and versioning the data for us 📦

# Scorers

Now let's define the scorers we'll use to evaluate our model's responses. We use two complementary approaches: one simple and fast, and one that uses an LLM as a judge.

## Start Simple

Not only is it faster to start with simple functional checks, it also keeps costs down since no LLM call is needed to run them.  
This can test things such as:
- **Response length**: is the output within a character limit?
- **Structural checks**: is the output valid JSON, or does it contain required fields?
- **Behavioral signals**: does the model return the expected keyword or follow a format?

Our use case is a summarizer, so a natural first check is: **is the response shorter than the expected character limit?**

## LLM as Judge 🧑‍⚖️

Next, we use `make_judge` from MLflow to create an LLM-as-judge scorer.  
We provide it with instructions describing what a good response looks like, and it evaluates each generated answer with a `yes`/`no` verdict.  
We use our Llama 3.2 model as both the responder and the judge, meaning it is evaluating its own responses (💸💸💸).

In [ ]:
@scorer
def is_concise(outputs: str, expectations: dict) -> bool:
    """Is the response under the character limit?"""
    if not outputs:
        return False
    return len(outputs) <= expectations.get("length", 300)

os.environ["OPENAI_API_KEY"] = "no-key-required"

summary_quality_judge = make_judge(
    name="summary_quality",
    instructions=(
        "Evaluate the quality of the GENERATED_RESPONSE given the CONVERSATION history.\n\n"
        "{{ inputs }}\n"
        "{{ outputs }}\n\n"
        "Is the response a concise, accurate, and helpful reply to the conversation?"
    ),
    feedback_value_type=Literal["yes", "no"],
    model="openai:/llama32",
    base_url="http://llama-32-predictor.ai501.svc.cluster.local:80/v1/chat/completions",
    extra_headers={"Authorization": "Bearer no-key-required"},
)


print("✓ Scorers defined: summary_quality, is_concise")

## Run evaluation

We run the evaluation against the backend endpoint to make sure that we evaluate against the exact prompt and code we have in the backend

In [ ]:
def send_request(payload, url):
    import httpx
    import json
    full_response = ""

    with httpx.Client(timeout=None) as client:
        with client.stream("POST", url, json=payload) as response:
            for line in response.iter_lines():
                if line.startswith("data: "):
                    try:
                        data = json.loads(line[len("data: "):])
                        full_response += data.get("delta", "")
                    except json.JSONDecodeError:
                        continue

    return full_response

def predict_backend_summarize(messages: list, session_id: str = None, **kwargs) -> str:
      from urllib.parse import urljoin
      backend_url = "http://canopy-backend:8000"
      endpoint_to_test = "/summarize/chat"

      url = urljoin(backend_url, endpoint_to_test)
      payload = {
          "messages": messages,
          "session_id": session_id,
      }
      return send_request(payload, url)

In [ ]:
print("Running evaluation with our latest prompt...")
eval_v1 = mlflow.genai.evaluate(
    data=eval_dataset,      # ← using the MLflow-hosted dataset
    predict_fn=predict_backend_summarize,
    scorers=[is_concise, summary_quality_judge],
)

print("\n── Evaluation Results: Prompt v1 ──")
for metric, value in sorted(eval_v1.metrics.items()):
    bar = "█" * int((value if isinstance(value, float) else 0) * 20)
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}  {bar}")

print(f"\n✓ Results saved. View per-example breakdowns in the Evaluations tab.")
print(f"  → {MLFLOW_TRACKING_URI}")

To summarize, we have now used MLflow's evaluation framework to evaluate our backend endpoint using both a simple functional scorer and an LLM as judge.  
The results are tracked in MLflow, so you can compare evaluation runs over time and see exactly where performance improves or regresses across prompt or model changes.  
With this setup, we can build an evaluation workflow that lets us test our backend and model before it goes into production 👏